# Building a Retrieval-Augmented Generation (RAG) pipeline using PDF text chunking, HuggingFace embeddings, Chroma vector store, and OpenAI API.

In [ ]:
!pip install -q langchain-community pdfplumber langchain-text-splitters langchain-huggingface

In [ ]:
!pip install -q langchain-chroma

In [ ]:
!pip install openai


In [ ]:
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from openai import OpenAI

In [ ]:
doc = PDFPlumberLoader('Example.pdf')

In [ ]:
data = doc.load()
data

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_documents(data)

print(chunks)

In [ ]:

model_name = "sentence-transformers/all-mpnet-base-v2"
encoder = HuggingFaceEmbeddings(
    model_name=model_name
)

In [ ]:
string_chunks = [chunk.page_content for chunk in chunks]
encoder.embed_documents(string_chunks)

In [ ]:
vector_db = Chroma(
    embedding_function=encoder,
    persist_directory="db",
    collection_name="vectorDB"
)

In [ ]:
vector_db.add_documents(chunks)

In [ ]:
vector_db.search(query=" fox's place at sunset", search_type="similarity", k=1)

In [ ]:
vector_db.similarity_search_with_score(query="who is the author")

In [ ]:
 vector_db.get_by_ids(['143bf488-790a-40a0-b7f1-2acc64e80571'])

In [ ]:
 vector_db.delete(['143bf488-790a-40a0-b7f1-2acc64e80571'])

In [ ]:
from google.colab import userdata
api = userdata.get('api')


In [ ]:
from google.colab import userdata
url = userdata.get('base_url')

In [ ]:
a = input("Enter user query: ")

In [ ]:
b = vector_db.search(query= a, search_type="similarity", k=1)

In [ ]:
client = OpenAI(
    api_key = api,
    base_url = url
)

In [ ]:
response = client.chat.completions.create(
    model="google/gemini-2.5-flash",
    messages=[
        {"role": "user", "content": f"Here is the database information: {b}. Based on that, please answer this: {a}"}
    ],
    max_tokens=500
)
print(response.choices[0].message.content)